In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# Verifies numerical equivalence of AnalyteTransformerDecoder.embed()
# with flash_compatible=False (the default, used by every existing
# casanovo caller) against a standalone reimplementation of the exact
# PRE-PATCH original logic. No NAR patch is applied anywhere in this
# notebook — we are testing stock AR behavior only.
# ═══════════════════════════════════════════════════════════════════════
import os, time, inspect
import torch
import numpy as np

WORK_DIR    = '/teamspace/studios/this_studio/nar_profiling'
RESULTS_DIR = '/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results'
os.chdir(WORK_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

import depthcharge
print(f'depthcharge location: {depthcharge.__file__}')
if 'site-packages' in depthcharge.__file__:
    raise RuntimeError('Editable install not active — run pip install -e in terminal first.')
print('✓ Editable install (modified branch) confirmed\n')

from depthcharge.transformers import AnalyteTransformerDecoder
from depthcharge import utils as dc_utils

# Confirm flash_compatible parameter exists and defaults to False
_sig = inspect.signature(AnalyteTransformerDecoder.embed)
assert 'flash_compatible' in _sig.parameters, 'flash_compatible param missing!'
assert _sig.parameters['flash_compatible'].default is False, 'Default must be False for backward compat!'
print(f'flash_compatible parameter present, default={_sig.parameters["flash_compatible"].default} ✓')
print(f'Device: {DEVICE}')

FileNotFoundError: [Errno 2] No such file or directory: '/teamspace/studios/this_studio/nar_profiling'

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Load casanovo model in its natural AR configuration
# Deliberately does NOT apply the NAR monkey-patch from other notebooks.
# We are testing the model exactly as every existing casanovo user runs it.
# ═══════════════════════════════════════════════════════════════════════
from pathlib import Path
from casanovo.denovo import ModelRunner
from casanovo.config import Config
from casanovo.casanovo import _get_model_weights
import appdirs

config = Config(None)
cache_dir = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
model_path = _get_model_weights(cache_dir)
print(f'Model checkpoint: {model_path}')

runner = ModelRunner(config, model_path)
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model = runner.model.eval().to(DEVICE)

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')
print(f'Decoder: {type(model.decoder).__name__} (unpatched — stock AR behavior)')

# Sanity check: confirm this is the ORIGINAL, unpatched embed method
assert 'flash_compatible' in inspect.signature(model.decoder.embed).parameters
print('Decoder using modified depthcharge with flash_compatible param ✓')
print('No NAR patch applied — decoder.embed is the stock library method.\n')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — Standalone reference: EXACT pre-patch original embed() logic
# Copied verbatim from the original analytes.py source (before our
# flash_compatible parameter was added). Operates on the SAME decoder
# instance (same weights, same submodules) so any difference in output
# can only come from logic differences, not weight differences.
# ═══════════════════════════════════════════════════════════════════════
def reference_ar_embed(decoder, tokens, *args,
                        memory,
                        memory_key_padding_mask=None,
                        memory_mask=None,
                        tgt_mask=None,
                        **kwargs):
    """Byte-for-byte reimplementation of the ORIGINAL (pre-flash_compatible)
    AnalyteTransformerDecoder.embed() method, exactly as it existed before
    this PR. No flash_compatible logic exists here at all — this is the
    ground truth for what every existing casanovo user has always gotten.
    """
    if tokens is None:
        tokens = torch.tensor([[]]).to(decoder.device)

    encoded = decoder.token_encoder(tokens)
    global_token = decoder.global_token_hook(tokens, *args, **kwargs)
    encoded = torch.cat([global_token[:, None, :], encoded], dim=1)

    tgt_key_padding_mask = encoded.sum(axis=2) == 0
    tgt_key_padding_mask[:, 0] = False

    encoded = decoder.positional_encoder(encoded)

    if tgt_mask is None:
        tgt_mask = dc_utils.generate_tgt_mask(encoded.shape[1]).to(decoder.device)

    return decoder.transformer_decoder(
        tgt=encoded,
        memory=memory,
        tgt_mask=tgt_mask,
        tgt_key_padding_mask=tgt_key_padding_mask,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
    )

print('Reference implementation defined (pre-patch original logic) ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Real test inputs: actual spectra + actual tokenized peptides
# of VARYING lengths in the same batch, so tgt_key_padding_mask (the
# code our patch touches) is genuinely exercised, not trivially all-False.
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule

MGF_FILE   = 'multi-enzyme-simple.test.mgf'
SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = '.lance_cache'

_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=8, tokenizer=runner.tokenizer,
    max_charge=getattr(model, 'max_charge', 10), n_workers=0,
)
_dm.setup(stage='test', annotated=False)
_batch = next(iter(_dm.predict_dataloader()))
mzs, ints, precursors, _ = model._process_batch(_batch)
mzs, ints, precursors = mzs.to(DEVICE), ints.to(DEVICE), precursors.to(DEVICE)

with torch.no_grad():
    memory, mem_mask = model.encoder(mzs, ints)
print(f'Encoder output: memory={memory.shape}  mem_mask={mem_mask.shape}')

# Real tokenized peptides of DIFFERENT lengths → genuine padding in batch
peptides = ['LESLIEK', 'PEPTIDER', 'EDITHYKK', 'AK', 'GAVLIMFWPSTCYNQDEKRH',
            'MSKQIVLK', 'SEQUENCE', 'A']
tokens = runner.tokenizer.tokenize(peptides).to(DEVICE)
print(f'Tokenized peptides: {tokens.shape}  (varying lengths → real padding)')
print(f'Token padding present: {(tokens == 0).any().item()}')

# Trim memory/mem_mask/precursors to match token batch size (8)
memory       = memory[:len(peptides)]
mem_mask     = mem_mask[:len(peptides)]
precursors   = precursors[:len(peptides)]
print(f'\nTest batch ready: {len(peptides)} sequences, lengths {[len(p) for p in peptides]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Run the NEW (patched) code with flash_compatible=False
# This is the exact call every existing casanovo installation makes —
# nobody passes flash_compatible explicitly, so this exercises the
# default branch of our modified embed()/forward().
# ═══════════════════════════════════════════════════════════════════════
with torch.no_grad():
    # Test 1: embed() directly, default tgt_mask=None (auto-generates causal mask)
    new_embed_auto_mask = model.decoder.embed(
        tokens, memory=memory, memory_key_padding_mask=mem_mask,
    )  # flash_compatible defaults to False — not passed explicitly

    # Test 2: embed() with an explicit caller-provided tgt_mask
    explicit_mask = dc_utils.generate_tgt_mask(tokens.shape[1] + 1).to(DEVICE)
    new_embed_explicit_mask = model.decoder.embed(
        tokens, memory=memory, memory_key_padding_mask=mem_mask,
        tgt_mask=explicit_mask,
    )

    # Test 3: full forward() → final scores (what casanovo actually uses)
    new_scores = model.decoder(
        tokens, memory=memory, memory_key_padding_mask=mem_mask,
    )

    # Test 4: tokens=None (start-of-sequence case, used during AR generation)
    new_embed_none_tokens = model.decoder.embed(
        None, memory=memory[:1], memory_key_padding_mask=mem_mask[:1],
    )

print('NEW code path (flash_compatible=False, default) executed:')
print(f'  embed (auto mask)     : {new_embed_auto_mask.shape}')
print(f'  embed (explicit mask) : {new_embed_explicit_mask.shape}')
print(f'  forward (scores)      : {new_scores.shape}')
print(f'  embed (tokens=None)   : {new_embed_none_tokens.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Run the REFERENCE (pre-patch original) logic on the identical
# inputs, using the SAME decoder instance (same weights).
# ═══════════════════════════════════════════════════════════════════════
with torch.no_grad():
    ref_embed_auto_mask = reference_ar_embed(
        model.decoder, tokens, memory=memory, memory_key_padding_mask=mem_mask,
    )
    ref_embed_explicit_mask = reference_ar_embed(
        model.decoder, tokens, memory=memory, memory_key_padding_mask=mem_mask,
        tgt_mask=explicit_mask,
    )
    ref_scores = model.decoder.score_embeddings(ref_embed_auto_mask)
    ref_embed_none_tokens = reference_ar_embed(
        model.decoder, None, memory=memory[:1], memory_key_padding_mask=mem_mask[:1],
    )

print('REFERENCE path (pre-patch original logic) executed:')
print(f'  embed (auto mask)     : {ref_embed_auto_mask.shape}')
print(f'  embed (explicit mask) : {ref_embed_explicit_mask.shape}')
print(f'  forward (scores)      : {ref_scores.shape}')
print(f'  embed (tokens=None)   : {ref_embed_none_tokens.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Compare NEW vs REFERENCE outputs numerically
# ═══════════════════════════════════════════════════════════════════════
def compare(name, a, b, atol=1e-6, rtol=1e-5):
    is_close   = torch.allclose(a, b, atol=atol, rtol=rtol)
    is_exact   = torch.equal(a, b)
    max_diff   = (a - b).abs().max().item()
    mean_diff  = (a - b).abs().mean().item()
    status = '✓ IDENTICAL' if is_exact else ('✓ allclose' if is_close else '✗ MISMATCH')
    print(f'{name:<28} {status:<14} exact={is_exact}  allclose={is_close}  '
          f'max_diff={max_diff:.2e}  mean_diff={mean_diff:.2e}')
    return is_exact, is_close, max_diff

print('── Numerical Equivalence: NEW (flash_compatible=False) vs REFERENCE ──\n')
results = []
results.append(compare('embed (auto mask)',     new_embed_auto_mask,     ref_embed_auto_mask))
results.append(compare('embed (explicit mask)', new_embed_explicit_mask, ref_embed_explicit_mask))
results.append(compare('forward (scores)',      new_scores,              ref_scores))
results.append(compare('embed (tokens=None)',   new_embed_none_tokens,   ref_embed_none_tokens))

all_exact    = all(r[0] for r in results)
all_allclose = all(r[1] for r in results)
print(f'\nAll outputs bit-exact  : {"✓ YES" if all_exact else "✗ NO"}')
print(f'All outputs allclose   : {"✓ YES" if all_allclose else "✗ NO"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Repeat the comparison across multiple random batches, batch
# sizes, and seeds to rule out a single lucky match.
# ═══════════════════════════════════════════════════════════════════════
sweep_results = []
_dm_sweep = DeNovoDataModule(
    lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=16, tokenizer=runner.tokenizer,
    max_charge=getattr(model, 'max_charge', 10), n_workers=0,
)
_dm_sweep.setup(stage='test', annotated=False)
_loader = iter(_dm_sweep.predict_dataloader())

test_peptide_pool = [
    'LESLIEK', 'PEPTIDER', 'EDITHYKK', 'AK', 'GAVLIMFWPSTCYNQDEKRH',
    'MSKQIVLK', 'SEQUENCE', 'A', 'PROTEIN', 'CASANOVOTEST',
]

print('── Robustness sweep: 10 random batches, varying batch size & seed ──\n')
for trial in range(10):
    torch.manual_seed(trial)
    try:
        _b = next(_loader)
    except StopIteration:
        _loader = iter(_dm_sweep.predict_dataloader())
        _b = next(_loader)
    _mz, _it, _pr, _ = model._process_batch(_b)
    _mz, _it, _pr = _mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE)

    bs = min(len(test_peptide_pool), _mz.shape[0])
    peps = test_peptide_pool[:bs]
    toks = runner.tokenizer.tokenize(peps).to(DEVICE)

    with torch.no_grad():
        mem, mmk = model.encoder(_mz[:bs], _it[:bs])
        new_out = model.decoder(toks, memory=mem, memory_key_padding_mask=mmk)
        ref_emb = reference_ar_embed(model.decoder, toks, memory=mem, memory_key_padding_mask=mmk)
        ref_out = model.decoder.score_embeddings(ref_emb)

    is_exact = torch.equal(new_out, ref_out)
    max_diff = (new_out - ref_out).abs().max().item()
    sweep_results.append((trial, bs, is_exact, max_diff))
    print(f'  Trial {trial}: bs={bs:2d}  exact={is_exact}  max_diff={max_diff:.2e}')

all_sweep_exact = all(r[2] for r in sweep_results)
print(f'\nAll {len(sweep_results)} sweep trials bit-exact: {"✓ YES" if all_sweep_exact else "✗ NO"}')